In [2]:
%load_ext sql

In [3]:
%sql sqlite:///employee_details.db

Connecting to 'sqlite:///employee_details.db'

In [4]:
%%sql
CREATE TABLE IF NOT EXISTS Employee (
    EmployeeID INTEGER PRIMARY KEY,
    FirstName TEXT,
    LastName TEXT,
    Department TEXT,
    Salary REAL,
    HireDate TEXT
);

Running query in 'sqlite:///employee_details.db'

++
||
++
++

In [5]:
import sqlite3
import random

conn = sqlite3.connect("employee_details.db")
cur = conn.cursor()

first_names = ["John","Jane","Michael","Sarah","David","Emma","Chris","Anna","James","Olivia"]
last_names = ["Smith","Johnson","Lee","Brown","Davis","Miller","Wilson","Taylor","Anderson","Thomas"]
departments = ["HR","IT","Finance","Marketing","Sales"]

for i in range(1, 31):
    cur.execute("""
        INSERT INTO Employee (EmployeeID, FirstName, LastName, Department, Salary, HireDate)
        VALUES (?, ?, ?, ?, ?, ?)
    """, (
        i,
        random.choice(first_names),
        random.choice(last_names),
        random.choice(departments),
        round(random.uniform(30000, 120000), 2),
        f"202{random.randint(0,5)}-{random.randint(1,12):02d}-{random.randint(1,28):02d}"
    ))

conn.commit()
conn.close()

In [7]:
%sql select * from Employee

Running query in 'sqlite:///employee_details.db'

EmployeeID,FirstName,LastName,Department,Salary,HireDate
1,James,Brown,HR,60907.84,2023-06-01
2,Olivia,Miller,Marketing,85000.16,2025-09-18
3,James,Brown,Sales,118981.11,2025-01-14
4,Sarah,Davis,Sales,96161.56,2022-12-05
5,James,Thomas,HR,54615.47,2023-07-13
6,Michael,Anderson,HR,31221.35,2025-01-11
7,David,Wilson,Finance,118473.61,2023-06-10
8,David,Davis,HR,46625.22,2021-01-15
9,Anna,Taylor,Finance,57479.36,2025-08-11
10,Jane,Brown,HR,78734.63,2025-06-10


## 1. Filter employees by multiple departments
Return all employees who work in IT, Finance, or HR.

In [20]:
##1
%sql select * from employee where Department in ('HR', 'Finance', 'IT')

Running query in 'sqlite:///employee_details.db'

EmployeeID,FirstName,LastName,Department,Salary,HireDate
1,James,Brown,HR,60907.84,2023-06-01
5,James,Thomas,HR,54615.47,2023-07-13
6,Michael,Anderson,HR,31221.35,2025-01-11
7,David,Wilson,Finance,118473.61,2023-06-10
8,David,Davis,HR,46625.22,2021-01-15
9,Anna,Taylor,Finance,57479.36,2025-08-11
10,Jane,Brown,HR,78734.63,2025-06-10
15,Sarah,Davis,IT,31631.29,2022-09-03
16,Emma,Miller,Finance,111975.11,2025-12-22
17,John,Lee,Finance,90818.16,2020-01-15


## 2. Salary range filtering
Find employees whose salary is between 50,000 and 90,000.

In [24]:
%sql select * from employee where salary between 50000 and 90000

Running query in 'sqlite:///employee_details.db'

EmployeeID,FirstName,LastName,Department,Salary,HireDate
1,James,Brown,HR,60907.84,2023-06-01
2,Olivia,Miller,Marketing,85000.16,2025-09-18
5,James,Thomas,HR,54615.47,2023-07-13
9,Anna,Taylor,Finance,57479.36,2025-08-11
10,Jane,Brown,HR,78734.63,2025-06-10
12,Emma,Taylor,Marketing,63537.39,2024-06-08
13,Michael,Taylor,Marketing,60915.52,2025-09-21
19,Olivia,Brown,IT,57309.12,2022-09-03
23,Sarah,Anderson,HR,57386.46,2020-09-12
28,John,Miller,Sales,57958.04,2024-12-11


## 3. Extract hire year
Show:
FirstName
LastName
HireDate
Year hired (derived from HireDate)

In [29]:
%sql select FirstName, LastName, HireDate, substring(HireDate,1, 4) as [HireYear] from employee

Running query in 'sqlite:///employee_details.db'

FirstName,LastName,HireDate,HireYear
James,Brown,2023-06-01,2023
Olivia,Miller,2025-09-18,2025
James,Brown,2025-01-14,2025
Sarah,Davis,2022-12-05,2022
James,Thomas,2023-07-13,2023
Michael,Anderson,2025-01-11,2025
David,Wilson,2023-06-10,2023
David,Davis,2021-01-15,2021
Anna,Taylor,2025-08-11,2025
Jane,Brown,2025-06-10,2025


## 4. Average salary per department
Compute:
Department
Average Salary
Sort from highest to lowest average salary.

In [39]:
%sql select Department, round(avg(Salary),2) as [AverageSalary] from employee group by Department order by AverageSalary desc

Running query in 'sqlite:///employee_details.db'

Department,AverageSalary
Finance,97419.54
Sales,89484.84
Marketing,75118.04
HR,65334.11
IT,63908.89


## 5. Employee count per department
Show:
Department
Number of employees
Only include departments with more than 5 employees.

In [40]:
%sql select Department, Count(distinct employeeid) as [TotalEmployees] from employee group by department having TotalEmployees>5

Running query in 'sqlite:///employee_details.db'

Department,TotalEmployees
Finance,6
HR,10
Sales,7


## 6. Employees hired in recent years
Get employees hired in 2023 or later.

In [43]:
%sql select * from employee where hiredate>='2023-01-01'

Running query in 'sqlite:///employee_details.db'

EmployeeID,FirstName,LastName,Department,Salary,HireDate
1,James,Brown,HR,60907.84,2023-06-01
2,Olivia,Miller,Marketing,85000.16,2025-09-18
3,James,Brown,Sales,118981.11,2025-01-14
5,James,Thomas,HR,54615.47,2023-07-13
6,Michael,Anderson,HR,31221.35,2025-01-11
7,David,Wilson,Finance,118473.61,2023-06-10
9,Anna,Taylor,Finance,57479.36,2025-08-11
10,Jane,Brown,HR,78734.63,2025-06-10
12,Emma,Taylor,Marketing,63537.39,2024-06-08
13,Michael,Taylor,Marketing,60915.52,2025-09-21


## 7. Top 2 highest salaries per department
Return:
FirstName
LastName
Department
Salary
But only the top 2 earners per department.

In [ ]:
%sql select * from (select FirstName,Lastname, Department, Salary, rank() over (partition by Department order by Salary desc) as [DepartmentRank] from employee) where DepartmentRank <=2

Running query in 'sqlite:///employee_details.db'

FirstName,Lastname,Department,Salary,DepartmentRank
David,Wilson,Finance,118473.61,1
Emma,Miller,Finance,111975.11,2
Chris,Anderson,HR,103494.03,1
Sarah,Johnson,HR,101344.79,2
Olivia,Brown,IT,102786.27,1
Olivia,Brown,IT,57309.12,2
Chris,Johnson,Marketing,91019.07,1
Olivia,Miller,Marketing,85000.16,2
James,Brown,Sales,118981.11,1
Chris,Anderson,Sales,116069.05,2


## 8. Salary ranking within department
Rank employees within each department by salary (highest to lowest).
Show:
FirstName
Department
Salary
Rank

In [61]:
%sql select FirstName,Lastname, Department, Salary, rank() over (partition by Department order by Salary desc) as [DepartmentRank] from employee

Running query in 'sqlite:///employee_details.db'

FirstName,LastName,Department,Salary,DepartmentRank
David,Wilson,Finance,118473.61,1
Emma,Miller,Finance,111975.11,2
James,Johnson,Finance,104875.92,3
Olivia,Anderson,Finance,100895.06,4
John,Lee,Finance,90818.16,5
Anna,Taylor,Finance,57479.36,6
Chris,Anderson,HR,103494.03,1
Sarah,Johnson,HR,101344.79,2
Sarah,Miller,HR,87284.99,3
Jane,Brown,HR,78734.63,4


## 9. Department salary statistics report
For each department, show:
Min salary
Max salary
Average salary
Salary range (max - min)

In [78]:
%sql select department, min(salary) as [MinSalary], max(salary) as [MaxSalary], round(avg(salary),2) as [AvgSalary], cast(max(Salary) as text)|| " - " || cast(min(Salary) as text) as [Max-Min]from employee group by Department

Running query in 'sqlite:///employee_details.db'

Department,MinSalary,MaxSalary,AvgSalary,Max-Min
Finance,57479.36,118473.61,97419.54,118473.61 - 57479.36
HR,31221.35,103494.03,65334.11,103494.03 - 31221.35
IT,31631.29,102786.27,63908.89,102786.27 - 31631.29
Marketing,60915.52,91019.07,75118.04,91019.07 - 60915.52
Sales,34821.27,118981.11,89484.84,118981.11 - 34821.27


## 10. Above department average earners
Find employees whose salary is greater than their department’s average salary.

In [85]:
%sql select * from (select EmployeeID,Firstname,lastname, department, salary, round(avg(salary) over (partition by department),2) as [DeptAvgSalary] from employee) where salary>=deptavgsalary

Running query in 'sqlite:///employee_details.db'

EmployeeID,Firstname,lastname,department,salary,DeptAvgSalary
7,David,Wilson,Finance,118473.61,97419.54
16,Emma,Miller,Finance,111975.11,97419.54
20,Olivia,Anderson,Finance,100895.06,97419.54
30,James,Johnson,Finance,104875.92,97419.54
10,Jane,Brown,HR,78734.63,65334.11
24,Chris,Anderson,HR,103494.03,65334.11
26,Sarah,Johnson,HR,101344.79,65334.11
29,Sarah,Miller,HR,87284.99,65334.11
27,Olivia,Brown,IT,102786.27,63908.89
2,Olivia,Miller,Marketing,85000.16,75118.04
